In [2]:
import duckdb

# Creates a database file in your Olist folder (or opens it if it exists)
con = duckdb.connect("olist.duckdb")

tables = {
    "customers":      "olist_customers_dataset.csv",
    "geolocation":    "olist_geolocation_dataset.csv",
    "order_items":    "olist_order_items_dataset.csv",
    "payments":       "olist_order_payments_dataset.csv",
    "reviews":        "olist_order_reviews_dataset.csv",
    "orders":         "olist_orders_dataset.csv",
    "products":       "olist_products_dataset.csv",
    "sellers":        "olist_sellers_dataset.csv",
    "category_names": "product_category_name_translation.csv",
}

for name, file in tables.items():
    con.execute(f"CREATE OR REPLACE TABLE {name} AS SELECT * FROM read_csv_auto('data/{file}')")
    count = con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
    print(f"{name:<16}{count:>12,}")

customers             99,441
geolocation        1,000,163
order_items          112,650
payments             103,886
reviews               99,224
orders                99,441
products              32,951
sellers                3,095
category_names            71


In [3]:
con.close()

In [5]:
import duckdb
con = duckdb.connect("olist.duckdb")

con.sql("DESCRIBE orders").df()

,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,customer_id,VARCHAR,YES,None,None,None
2,order_status,VARCHAR,YES,None,None,None
3,order_purchase_timestamp,TIMESTAMP,YES,None,None,None
4,order_approved_at,TIMESTAMP,YES,None,None,None
5,order_delivered_carrier_date,TIMESTAMP,YES,None,None,None
6,order_delivered_customer_date,TIMESTAMP,YES,None,None,None
7,order_estimated_delivery_date,TIMESTAMP,YES,None,None,None


In [22]:
import pandas as pd
pd.set_option("display.float_format", "{:,.2f}".format)

In [6]:
con.sql("SELECT * FROM orders LIMIT 5").df()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [7]:
con.sql("""
    SELECT order_status, COUNT(*) AS orders
    FROM orders
    GROUP BY order_status
    ORDER BY orders DESC
""").df()

,order_status,orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [8]:
con.sql("""
    SELECT
        COUNT(*)                 AS total_rows,
        COUNT(DISTINCT order_id) AS unique_orders
    FROM orders
""").df()

,total_rows,unique_orders
0,99441,99441


In [9]:
con.sql("""
    SELECT
        COUNT(*) - COUNT(order_approved_at)             AS missing_approved,
        COUNT(*) - COUNT(order_delivered_carrier_date)  AS missing_carrier,
        COUNT(*) - COUNT(order_delivered_customer_date) AS missing_delivered,
        COUNT(*) - COUNT(order_estimated_delivery_date) AS missing_estimate
    FROM orders
""").df()

,missing_approved,missing_carrier,missing_delivered,missing_estimate
0,160,1783,2965,0


In [10]:
con.sql("""
    SELECT COUNT(*) AS delivered_but_no_date
    FROM orders
    WHERE order_status = 'delivered'
      AND order_delivered_customer_date IS NULL
""").df()

,delivered_but_no_date
0,8


In [11]:
con.sql("""
    SELECT
        order_status,
        COUNT(*)                             AS orders,
        COUNT(order_delivered_customer_date) AS has_delivery_date
    FROM orders
    GROUP BY order_status
    ORDER BY orders DESC
""").df()

,order_status,orders,has_delivery_date
0,delivered,96478,96470
1,shipped,1107,0
2,canceled,625,6
3,unavailable,609,0
4,invoiced,314,0
5,processing,301,0
6,created,5,0
7,approved,2,0


## Orders table: findings
- 99,441 orders, no duplicate order_ids
- ~97% delivered; delivery analysis will filter to delivered orders
- 8 delivered orders have no delivery date → excluded from delivery analysis
- 6 canceled orders have a delivery date (likely canceled after arrival); excluded by the delivered-only filter

In [12]:
con.sql("""
    SELECT
        COUNT(*)                  AS total_reviews,
        COUNT(DISTINCT review_id) AS unique_review_ids,
        COUNT(DISTINCT order_id)  AS orders_with_review
    FROM reviews
""").df()

,total_reviews,unique_review_ids,orders_with_review
0,99224,98410,98673


In [13]:
con.sql("""
    SELECT
        review_score,
        COUNT(*) AS reviews,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM reviews
    GROUP BY review_score
    ORDER BY review_score
""").df()

,review_score,reviews,pct
0,1,11424,11.5
1,2,3151,3.2
2,3,8179,8.2
3,4,19142,19.3
4,5,57328,57.8


In [14]:
con.sql("""
    SELECT n_reviews, COUNT(*) AS orders
    FROM (
        SELECT order_id, COUNT(*) AS n_reviews
        FROM reviews
        GROUP BY order_id
    )
    GROUP BY n_reviews
    ORDER BY n_reviews
""").df()

,n_reviews,orders
0,1,98126
1,2,543
2,3,4


In [15]:
con.sql("""
    SELECT order_id, review_score, review_creation_date
    FROM reviews
    WHERE order_id IN (
        SELECT order_id
        FROM reviews
        GROUP BY order_id
        HAVING COUNT(*) > 1
    )
    ORDER BY order_id, review_creation_date
    LIMIT 10
""").df()

,order_id,review_score,review_creation_date
0,0035246a40f520710769010f752e7507,5,2017-08-25
1,0035246a40f520710769010f752e7507,5,2017-08-29
2,013056cfe49763c6f66bda03396c5ee3,5,2018-02-22
3,013056cfe49763c6f66bda03396c5ee3,4,2018-03-04
4,0176a6846bcb3b0d3aa3116a9a768597,5,2017-12-30
5,0176a6846bcb3b0d3aa3116a9a768597,5,2017-12-30
6,02355020fd0a40a0d56df9f6ff060413,3,2018-03-21
7,02355020fd0a40a0d56df9f6ff060413,1,2018-03-29
8,029863af4b968de1e5d6a82782e662f5,5,2017-07-14
9,029863af4b968de1e5d6a82782e662f5,4,2017-07-19


In [16]:
con.sql("""
    CREATE OR REPLACE VIEW reviews_clean AS
    SELECT order_id, review_id, review_score, review_creation_date
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY order_id
                ORDER BY review_creation_date DESC, review_answer_timestamp DESC
            ) AS rn
        FROM reviews
    )
    WHERE rn = 1
""")

In [17]:
con.sql("""
    SELECT
        COUNT(*)                 AS total_rows,
        COUNT(DISTINCT order_id) AS unique_orders
    FROM reviews_clean
""").df()

,total_rows,unique_orders
0,98673,98673


## Reviews table: findings
- 547 orders have multiple reviews (customers can update reviews later); kept the most recent per order → `reviews_clean` view
- ~770 orders have no review
- Scores are polarized: 77% are 4–5 stars, but 1-star (11.5%) is as common as 2- and 3-star combined

In [18]:
con.sql("""
    SELECT
        COUNT(*)                 AS total_rows,
        COUNT(DISTINCT order_id) AS orders_with_items,
        MAX(order_item_id)       AS max_items_in_one_order
    FROM order_items
""").df()

,total_rows,orders_with_items,max_items_in_one_order
0,112650,98666,21


In [19]:
con.sql("""
    SELECT
        payment_type,
        COUNT(*)                 AS payment_rows,
        COUNT(DISTINCT order_id) AS orders
    FROM payments
    GROUP BY payment_type
    ORDER BY payment_rows DESC
""").df()

,payment_type,payment_rows,orders
0,credit_card,76795,76505
1,boleto,19784,19784
2,voucher,5775,3866
3,debit_card,1529,1528
4,not_defined,3,3


In [23]:
con.sql("""
    SELECT
        SUM(oi.price + oi.freight_value) AS item_revenue,
        SUM(p.payment_value)             AS payment_total
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN payments p     ON o.order_id = p.order_id
""").df()

,item_revenue,payment_total
0,"16,566,543.85","20,308,134.71"


In [24]:
con.sql("""
    WITH items AS (
        SELECT order_id, SUM(price + freight_value) AS item_total
        FROM order_items
        GROUP BY order_id
    ),
    pays AS (
        SELECT order_id, SUM(payment_value) AS payment_total
        FROM payments
        GROUP BY order_id
    )
    SELECT
        SUM(i.item_total)    AS item_revenue,
        SUM(p.payment_total) AS payment_total
    FROM items i
    JOIN pays p ON i.order_id = p.order_id
""").df()

,item_revenue,payment_total
0,"15,843,409.78","15,846,280.17"


## Order items & payments: findings
- Most orders have 1 item (avg ~1.14; max 21); ~775 orders have no items
- Vouchers are frequently split across multiple payment rows
- Joining items and payments together inflates revenue (payments overstated ~28%) due to fan-out → always aggregate each to order level before joining
- After aggregating, item revenue and payments reconcile within 0.02%

In [25]:
con.sql("""
    SELECT
        COUNT(*)                           AS total_rows,
        COUNT(DISTINCT customer_id)        AS distinct_customer_ids,
        COUNT(DISTINCT customer_unique_id) AS distinct_people
    FROM customers
""").df()

,total_rows,distinct_customer_ids,distinct_people
0,99441,99441,96096


In [26]:
con.sql("""
    WITH orders_per_person AS (
        SELECT c.customer_unique_id, COUNT(*) AS n_orders
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        GROUP BY c.customer_unique_id
    )
    SELECT
        n_orders,
        COUNT(*) AS people,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM orders_per_person
    GROUP BY n_orders
    ORDER BY n_orders
""").df()

,n_orders,people,pct
0,1,93099,96.88
1,2,2745,2.86
2,3,203,0.21
3,4,30,0.03
4,5,8,0.01
5,6,6,0.01
6,7,3,0.00
7,9,1,0.00
8,17,1,0.00


## Customers table: findings
- customer_id is unique per order; customer_unique_id identifies real people → use customer_unique_id for all retention analysis
- 96,096 unique customers across 99,441 orders
- 96.9% of customers ordered only once; ~3.1% are repeat buyers (mostly 2 orders)
- Implication: Olist relies almost entirely on new-customer acquisition; retention is the big opportunity